# FidelProbe: does a small frontier VLM understand the consonant × vowel structure of Ethiopic script?

1. **Runtime → Change runtime type → T4 GPU.**
2. Add your Hugging Face **write** token as a Colab secret: click the 🔑 key icon in the left sidebar → *Add new secret* → name `HF_TOKEN`, value `hf_...` → enable *Notebook access*.
3. **Runtime → Run all.** The full run of `Qwen/Qwen3-VL-2B-Instruct` on 2,592 items takes about 15–25 min on a T4; the last cell creates a public Hugging Face Bucket and uploads everything.

Every Ethiopic item has a matched Hangul (Korean) control, so a gap cannot be explained by a generic difficulty of syllabic scripts.

In [ ]:
!pip -q install -U "transformers>=4.57" accelerate "huggingface_hub>=1.6"
!apt-get -qq install -y fonts-noto-core fonts-noto-cjk fonts-sil-abyssinica > /dev/null
!git clone -q https://github.com/aleale2121/fatima_challenge.git
%cd fatima_challenge/fidelprobe
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

Rebuild the dataset (deterministic, seed 0). This reproduces `../data` exactly.

In [ ]:
!python build_dataset.py --out ../data --seed 0

## Run the model (resumable: rerun this cell if Colab disconnects)

In [ ]:
MODEL = "Qwen/Qwen3-VL-2B-Instruct"
!python run_eval.py --model $MODEL --data ../data --out ../results

## Tokenizer probe, scoring, tables and plots

In [ ]:
!python tokenizer_probe.py --model $MODEL --out ../results > /dev/null
!python analyze.py --data ../data --results ../results > /dev/null
from IPython.display import Image, Markdown, display
import glob
display(Markdown(open(glob.glob('../results/RESULTS_*.md')[0]).read().split('![')[0]))
for p in sorted(glob.glob('../results/fig_*.png')): display(Image(p))

## Look at individual failures

In [ ]:
import pandas as pd
df = pd.read_json(glob.glob('../results/scored_*.jsonl')[0], lines=True)
df[(df.script=='eth') & (~df.correct)][['task','gold','answer','raw']].sample(20, random_state=0)

## Publish: create the public Hugging Face Bucket and upload code, data, results, README and model weights

In [ ]:
!python upload_to_hf.py --bucket fidelprobe --model $MODEL